# Week 50, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w050d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
edges = [(1, 2), (2, 3), (3, 1), (4, 5)]
adj = {}
for a, b in edges:
    adj.setdefault(a, set()).add(b)
    adj.setdefault(b, set()).add(a)
print({v: len(n) for v, n in sorted(adj.items())}, sum(map(len, adj.values())) // 2)

**Predict before you run:** what will this print?



In [ ]:
from collections import deque

adj = {"a": ["b", "c"], "b": ["d"], "c": ["d", "e"], "d": ["f"], "e": ["f"], "f": []}
dist, q = {"a": 0}, deque(["a"])
while q:
    v = q.popleft()
    for w in adj[v]:
        if w not in dist:
            dist[w] = dist[v] + 1
            q.append(w)
print(dist)

**Predict before you run:** what will this print?



In [ ]:
adj = {1: [2, 3], 2: [4], 3: [4], 4: []}
order, seen = [], set()
def go(v):
    seen.add(v)
    for w in adj[v]:
        if w not in seen:
            go(w)
    order.append(v)
go(1)
print(order, order[::-1])

**Predict before you run:** what will this print?



In [ ]:
parent = {x: x for x in range(6)}
def find(x):
    while parent[x] != x:
        x = parent[x]
    return x
for a, b in [(0, 1), (2, 3), (1, 3), (4, 5)]:
    parent[find(a)] = find(b)
print(len({find(x) for x in range(6)}), find(0) == find(2))

**Predict before you run:** what will this print?



In [ ]:
from graphlib import TopologicalSorter

ts = TopologicalSorter({"tea": {"water", "cup"}, "water": {"kettle"}})
ts.prepare()
rounds = []
while ts.is_active():
    ready = sorted(ts.get_ready())
    rounds.append(ready)
    ts.done(*ready)
print(rounds)

### Exercise: Is it a tree?

Write `valid_tree(n, edges)`: whether the undirected graph on vertices 0 to n − 1 with these edges is a tree. Use union-find: a tree has exactly n − 1 edges, and no edge joins two vertices already connected.

In [ ]:
def valid_tree(n, edges):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_tree():\n    \"trees\"\n    assert valid_tree(5, [(0, 1), (0, 2), (0, 3), (1, 4)]) and not valid_tree(5, [(0, 1), (1, 2), (2, 3), (1, 3), (1, 4)])\n    assert valid_tree(1, []) and not valid_tree(2, []) and not valid_tree(4, [(0, 1), (2, 3), (1, 0)])\n    rng = random.Random(1)\n    for _ in range(200):\n        n = rng.randint(1, 7)\n        edges = list({tuple(sorted(rng.sample(range(n), 2))) for _ in range(rng.randint(0, 7))}) if n > 1 else []\n        adj = {v: [] for v in range(n)}\n        for a, b in edges:\n            adj[a].append(b)\n            adj[b].append(a)\n        seen, stack = {0}, [0]\n        while stack:\n            v = stack.pop()\n            for w in adj[v]:\n                if w not in seen:\n                    seen.add(w)\n                    stack.append(w)\n        assert valid_tree(n, edges) == (len(seen) == n and len(edges) == n - 1)")

### Exercise: A knight's journey

On an 8 by 8 chessboard, a knight moves two squares one way and one the other. Write `knight_moves(start, goal)`: the fewest moves between two squares given as `(row, col)` from 0 to 7, by BFS over the implicit graph of squares. Then `knight_map(start)`: an 8 by 8 list of lists of the fewest moves to every square.

In [ ]:
from collections import deque

def knight_moves(start, goal):
    ___

def knight_map(start):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_knight():\n    \"knights\"\n    assert knight_moves((0, 0), (0, 0)) == 0 and knight_moves((0, 0), (1, 2)) == 1 and knight_moves((0, 0), (7, 7)) == 6\n    assert knight_moves((0, 0), (1, 1)) == 4\n    m = knight_map((0, 0))\n    assert m[0][0] == 0 and m[2][1] == 1 and max(max(r) for r in m) == 6 and len(m) == 8 and all(len(r) == 8 for r in m)")

### Exercise: Forge: a transport network

Write `Network` to the specification above.

In [ ]:
from collections import deque

class Network:
    def __init__(self, lines):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nLINES = {\n    \"Red\": [\"Harbour\", \"Market\", \"Cathedral\", \"University\", \"Hospital\"],\n    \"Blue\": [\"Airport\", \"Station\", \"Cathedral\", \"Stadium\"],\n    \"Green\": [\"Station\", \"Park\", \"Zoo\"],\n    \"Yellow\": [\"Zoo\", \"Lakeside\"],\n    \"Grey\": [\"Mill\", \"Quarry\"],\n}\ndef test_basics():\n    \"stations and lines\"\n    n = Network(LINES)\n    assert n.stations()[:3] == [\"Airport\", \"Cathedral\", \"Harbour\"] and len(n.stations()) == 13\n    assert n.lines_at(\"Cathedral\") == [\"Blue\", \"Red\"] and n.lines_at(\"Zoo\") == [\"Green\", \"Yellow\"]\n    for bad in ({\"A\": [\"x\"]}, {\"A\": [\"x\", \"y\", \"x\"]}):\n        try:\n            Network(bad)\n            assert False\n        except ValueError:\n            pass\n    try:\n        n.route(\"Nowhere\", \"Zoo\")\n        assert False\n    except KeyError:\n        pass\ndef test_routes():\n    \"routes\"\n    n = Network(LINES)\n    assert n.route(\"Harbour\", \"Lakeside\") == [\"Harbour\", \"Market\", \"Cathedral\", \"Station\", \"Park\", \"Zoo\", \"Lakeside\"]\n    assert n.fewest_stops(\"Harbour\", \"Lakeside\") == 6 and n.fewest_stops(\"Mill\", \"Zoo\") is None and n.route(\"Zoo\", \"Zoo\") == [\"Zoo\"]\n    assert n.fewest_changes(\"Harbour\", \"Lakeside\") == 3 and n.fewest_changes(\"Market\", \"Hospital\") == 0\n    assert n.fewest_changes(\"Airport\", \"Zoo\") == 1 and n.fewest_changes(\"Mill\", \"Zoo\") is None and n.fewest_changes(\"Park\", \"Park\") == 0\n    assert n.within(\"Cathedral\", 1) == [\"Cathedral\", \"Market\", \"Stadium\", \"Station\", \"University\"]\ndef test_closures():\n    \"closures\"\n    n = Network(LINES)\n    parts = n.split_if_closed(\"Cathedral\")\n    assert [\"Harbour\", \"Market\"] in parts and [\"Hospital\", \"University\"] in parts and [\"Mill\", \"Quarry\"] in parts\n    assert len(parts) == 5\n    assert n.critical() == [\"Cathedral\", \"Market\", \"Park\", \"Station\", \"University\", \"Zoo\"]\ndef test_random():\n    \"random networks\"\n    rng = random.Random(1)\n    for _ in range(30):\n        names = [f\"s{i}\" for i in range(12)]\n        lines = {}\n        for k in range(rng.randint(1, 5)):\n            lines[f\"L{k}\"] = rng.sample(names, rng.randint(2, 5))\n        n = Network(lines)\n        for _ in range(10):\n            a, b = rng.choice(n.stations()), rng.choice(n.stations())\n            r = n.route(a, b)\n            if r is not None:\n                assert r[0] == a and r[-1] == b and all(y in n.adj[x] for x, y in zip(r, r[1:]))\n                assert len(r) - 1 == n.fewest_stops(a, b)\n                assert n.fewest_changes(a, b) is not None\n            else:\n                assert n.fewest_changes(a, b) is None")

> **The right graph for the question.** The same city gave two different graphs today. For **fewest stops**, stations are the vertices and track between them the edges. For **fewest changes**, lines are the vertices and shared stations the edges: a route with few changes may have many stops, and BFS on the station graph can't see changes at all. Choosing what the vertices and edges are is usually the real work in a graph problem; once the graph is right, the algorithm is often one you already know. (Week 49's word ladder did the same: words as vertices, one-letter changes as edges.) When a question doesn't fit, ask whether a different graph, with richer vertices such as "station and current line" or "cell and doors used", would make it a plain BFS.

### Exercise: Forge bonus: a package installer

A package index maps each package to the list of packages it depends on directly. Write `Installer(index)`, keeping `installed` (a set) and `explicit` (the packages installed by name, as opposed to pulled in as dependencies):

- `install_order(package)`: every package needed to install it (itself and all its dependencies, directly or not), in an order where each comes after its dependencies, found by DFS **post-order** visiting dependencies in their listed order, with an explicit stack (the checks use a chain of 5,000). Raise `KeyError` for an unknown package; `ValueError` naming a missing dependency; and `ValueError` with the cycle written as `a -> b -> c -> a` (starting from the first package of the cycle reached) if dependencies loop.
- `install(package)`: install everything in its order that isn't installed yet, mark the package explicit, and return the list of newly installed packages.
- `dependents(package)`: the installed packages that need it, directly or indirectly, sorted.
- `uninstall(package)`: `KeyError` if not installed; `ValueError` listing the dependents if any installed package still needs it; otherwise remove it.
- `autoremove()`: remove every installed package that no explicit package needs, returning them sorted.

In [ ]:
class Installer:
    def __init__(self, index):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("INDEX = {\n    \"webapp\": [\"router\", \"templates\", \"db\"],\n    \"router\": [\"utils\"],\n    \"templates\": [\"utils\", \"markup\"],\n    \"db\": [\"utils\"],\n    \"markup\": [],\n    \"utils\": [],\n    \"charts\": [\"maths\", \"colours\"],\n    \"maths\": [],\n    \"colours\": [],\n}\ndef test_order():\n    \"install order\"\n    inst = Installer(INDEX)\n    assert inst.install_order(\"webapp\") == [\"utils\", \"router\", \"markup\", \"templates\", \"db\", \"webapp\"]\n    assert inst.install_order(\"utils\") == [\"utils\"]\n    try:\n        inst.install_order(\"nothing\")\n        assert False\n    except KeyError:\n        pass\n    bad = Installer({\"a\": [\"b\"], \"b\": [\"c\"], \"c\": [\"a\"]})\n    try:\n        bad.install_order(\"a\")\n        assert False\n    except ValueError as e:\n        assert \"a -> b -> c -> a\" in str(e)\n    missing = Installer({\"a\": [\"zz\"]})\n    try:\n        missing.install_order(\"a\")\n        assert False\n    except ValueError as e:\n        assert \"zz\" in str(e)\ndef test_install():\n    \"installing and removing\"\n    inst = Installer(INDEX)\n    assert inst.install(\"templates\") == [\"utils\", \"markup\", \"templates\"]\n    assert inst.install(\"webapp\") == [\"router\", \"db\", \"webapp\"]\n    assert inst.dependents(\"utils\") == [\"db\", \"router\", \"templates\", \"webapp\"]\n    try:\n        inst.uninstall(\"utils\")\n        assert False\n    except ValueError as e:\n        assert \"webapp\" in str(e)\n    inst.uninstall(\"webapp\")\n    assert inst.autoremove() == [\"db\", \"router\"] and inst.installed == {\"utils\", \"markup\", \"templates\"}\n    inst.uninstall(\"templates\")\n    assert inst.autoremove() == [\"markup\", \"utils\"] and inst.installed == set()\ndef test_deep():\n    \"a long chain\"\n    index = {f\"p{i}\": [f\"p{i + 1}\"] for i in range(5000)}\n    index[\"p5000\"] = []\n    inst = Installer(index)\n    order = inst.install_order(\"p0\")\n    assert order[0] == \"p5000\" and order[-1] == \"p0\" and len(order) == 5001")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [representations](w050d1.html), [breadth-first search](w050d2.html), [depth-first search](w050d3.html), [connected components](w050d4.html) or [topological sort](w050d5.html).

## Mixed review

## Forge: a transport network

A city's transport network is a set of **lines**, each a list of stations in order; trains run both ways, and a station can be on several lines. Build a planner that answers the questions travellers and planners ask.

Write `Network(lines)`, where `lines` is a dict from line name to its list of stations:

- Raise `ValueError` for a line with fewer than two stations or one that visits a station twice.
- Store the station graph as `adj`, a dict from each station to the **set** of stations next to it on any line.
- `stations()` (sorted) and `lines_at(station)` (sorted line names). Any method given an unknown station raises `KeyError`.
- `route(a, b)`: a route with the fewest stops, as a list of stations (`[a]` when `a == b`; `None` if unreachable). Explore neighbours in **alphabetical** order, so the route is the one BFS finds that way. `fewest_stops(a, b)`: its number of stops (edges), or `None`.
- `within(a, k)`: every station at most k stops from a, sorted.
- `fewest_changes(a, b)`: the fewest times a traveller must change line (0 if one line serves both, or `a == b`; `None` if unreachable). Run BFS on a **different graph**: the lines are the vertices, and two lines are joined when they share a station. Start from every line at a (distance 0) and stop at the first line that serves b.
- `split_if_closed(station)`: if that station closed, the pieces the rest of the network would fall into, each a sorted list, the list sorted.
- `critical()`: the sorted stations whose closure would split their part of the network into more pieces (compare the number of pieces with and without the station). A simple approach that closes each station in turn is fine for networks of a few hundred stations. (A single DFS can find them all in O(V + E): they're called **articulation points**, and Robert Tarjan's 1972 algorithm finds them from discovery times.)

## Forge bonus: a package installer

## Weekly quiz

## Week 50 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can describe graphs precisely, store them three ways, choose between them, and work with implicit graphs. ([Day 1](w050d1.html))
- I can run BFS for distances and shortest paths, on grids, from many sources, and to test for two colours. ([Day 2](w050d2.html))
- I can run DFS recursively and with a stack, record finish times, detect cycles, and list paths by backtracking. ([Day 3](w050d3.html))
- I can find components and islands, and use union-find as edges arrive. ([Day 4](w050d4.html))
- I can sort topologically with Kahn's algorithm, with DFS and with graphlib, and find a critical path. ([Day 5](w050d5.html))
- I built a transport network planner and a package installer. (Today)

Tomorrow is a **rest day**. Next week: **Graphs II**, where edges get weights: shortest paths with Dijkstra's algorithm and Bellman-Ford, more union-find, minimum spanning trees, and grids as graphs.